In [7]:
!py -3.10 -m pip install tqdm
!py -3.10 -m pip install torch torchvision torchaudio



[notice] A new release of pip available: 22.3.1 -> 25.1.1
[notice] To update, run: C:\Users\PC\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip


     -------------------------------------- 212.5/212.5 MB 1.7 MB/s eta 0:00:00
     ---------------------------------------- 1.7/1.7 MB 1.7 MB/s eta 0:00:00
     ---------------------------------------- 2.5/2.5 MB 1.7 MB/s eta 0:00:00
     ---------------------------------------- 1.7/1.7 MB 1.7 MB/s eta 0:00:00
     ---------------------------------------- 6.3/6.3 MB 1.8 MB/s eta 0:00:00
     -------------------------------------- 134.9/134.9 kB 2.7 MB/s eta 0:00:00
     -------------------------------------- 194.4/194.4 kB 2.0 MB/s eta 0:00:00
     -------------------------------------- 536.2/536.2 kB 2.1 MB/s eta 0:00:00


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip available: 22.3.1 -> 25.1.1
[notice] To update, run: C:\Users\PC\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip


In [1]:
# =============================================================================
# 03_feature_extraction.py
# -----------------------------------------------------------------------------
# Extracts visual features from mouth-cropped frames using:
# 🔹 2D ResNet18 → appearance features from center frame
# 🔹 3D r3d_18    → spatio-temporal features from sliding window clips
#
# Input:
#   - mouth_crops/*.jpg (cropped & aligned mouth regions)
# Output:
#   - features.pt       (Tensor of concatenated 2D+3D features)
#
# Compatible with CPU (PyTorch fallback if GPU unavailable)
# =============================================================================

import os
import cv2
import numpy as np
from glob import glob
from tqdm import tqdm

import torch
import torchvision.transforms as transforms
from torchvision.models import resnet18, video

# ==========================================
# 📁 Setup Paths
# ==========================================
video_dir = r"C:\Users\PC\Desktop\phaseB\VisoSpeak\data\vid_001"
mouth_dir = os.path.join(video_dir, "mouth_crops")
feature_dir = os.path.join(video_dir, "features")
os.makedirs(feature_dir, exist_ok=True)

# ==========================================
# ⚙️ Preprocessing Transforms
# ==========================================
resnet_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

r3d_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((112, 112)),
    transforms.ToTensor()
])

# ==========================================
# 🧠 Load Models
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2D ResNet18 (outputs 512-dim features)
resnet = resnet18(pretrained=True)
resnet.fc = torch.nn.Identity()
resnet = resnet.to(device).eval()

# 3D R3D_18 (outputs 512-dim features)
r3d = video.r3d_18(pretrained=True)
r3d.fc = torch.nn.Identity()
r3d = r3d.to(device).eval()

# ==========================================
# 📥 Load Cropped Mouth Frames
# ==========================================
frame_paths = sorted(glob(os.path.join(mouth_dir, "*.jpg")))
print(f"Found {len(frame_paths)} cropped mouth frames.")

# ==========================================
# 📊 Feature Containers
# ==========================================
resnet_features = []
r3d_clip = []  # temporary clip of 16 frames
r3d_features = []

# ==========================================
# 🔄 Iterate over frames
# ==========================================
for frame_path in tqdm(frame_paths, desc="Extracting features"):
    img = cv2.imread(frame_path)
    if img is None:
        continue

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # === 2D ResNet Feature Extraction ===
    resnet_input = resnet_transform(img_rgb).unsqueeze(0).to(device)
    with torch.no_grad():
        res_feat = resnet(resnet_input).cpu().numpy()
    resnet_features.append(res_feat.squeeze())

    # === Collect clip for 3D R3D ===
    r3d_input = r3d_transform(img_rgb)
    r3d_clip.append(r3d_input)

    if len(r3d_clip) == 16:
        clip_tensor = torch.stack(r3d_clip).permute(1, 0, 2, 3).unsqueeze(0).to(device)  # (B, C, T, H, W)
        with torch.no_grad():
            r3d_feat = r3d(clip_tensor).cpu().numpy()
        r3d_features.append(r3d_feat.squeeze())
        r3d_clip = []  # reset for next clip

# ==========================================
# 💾 Save Features to Disk
# ==========================================
resnet_features = np.array(resnet_features)       # (N, 512)
r3d_features = np.array(r3d_features)             # (M, 512)

np.save(os.path.join(feature_dir, "resnet_features.npy"), resnet_features)
np.save(os.path.join(feature_dir, "r3d_features.npy"), r3d_features)

print(f"✅ Saved ResNet features: {resnet_features.shape}")
print(f"✅ Saved R3D features: {r3d_features.shape}")
print(f"📁 Output directory: {feature_dir}")


C:\Users\PC\AppData\Local\Programs\Python\Python310\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\PC\AppData\Local\Programs\Python\Python310\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
C:\Users\PC\AppData\Local\Programs\Python\Python310\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=R3D_18_Weights.KINETICS400

Found 75 cropped mouth frames.


Extracting features: 100%|█████████████████████████████████████████████████████████████| 75/75 [00:04<00:00, 17.17it/s]

✅ Saved ResNet features: (75, 512)
✅ Saved R3D features: (4, 512)
📁 Output directory: C:\Users\PC\Desktop\phaseB\VisoSpeak\data\vid_001\features
